# RAG Evaluation with LangChain and LangSmith

This notebook demonstrates how to build and evaluate a RAG-based medical chatbot using LangChain and LangSmith.

## Step 1: Install Dependencies

**IMPORTANT:** After running the cell below, you MUST restart the runtime:
- Go to **Runtime → Restart runtime** (or press Ctrl+M then .)
- Then continue from **Step 2**

In [ ]:
# Install all required packages with compatible versions
# Run this cell FIRST, then RESTART RUNTIME before continuing

!pip uninstall -y openai langchain langchain-core langchain-community langchain-openai langsmith chromadb httpx 2>/dev/null

!pip install -q \
    httpx==0.27.2 \
    openai==1.54.0 \
    langchain==0.3.7 \
    langchain-core==0.3.15 \
    langchain-community==0.3.5 \
    langchain-openai==0.2.6 \
    langsmith==0.1.140 \
    chromadb==0.5.15 \
    pandas \
    gdown

print("\n" + "="*60)
print("✅ Installation complete!")
print("="*60)
print("\n⚠️  IMPORTANT: You MUST restart the runtime now!")
print("   Go to: Runtime → Restart runtime")
print("   Then continue from Step 2")
print("="*60)

Found existing installation: openai 2.9.0
Uninstalling openai-2.9.0:
  Successfully uninstalled openai-2.9.0
Found existing installation: langchain 1.1.3
Uninstalling langchain-1.1.3:
  Successfully uninstalled langchain-1.1.3
Found existing installation: langchain-core 1.1.3
Uninstalling langchain-core-1.1.3:
  Successfully uninstalled langchain-core-1.1.3
Found existing installation: langsmith 0.4.56
Uninstalling langsmith-0.4.56:
  Successfully uninstalled langsmith-0.4.56
Found existing installation: httpx 0.28.1
Uninstalling httpx-0.28.1:
  Successfully uninstalled httpx-0.28.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reason for being yanked: Imported six in a critical path
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4

## Step 2: Set API Keys

⚠️ **Make sure you restarted the runtime after Step 1!**

Enter your OpenAI and LangSmith API keys below.

In [ ]:
import os
from getpass import getpass

# Option 1: Interactive input (recommended for security)
# Uncomment the lines below to enter keys interactively
# os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")
# os.environ["LANGSMITH_API_KEY"] = getpass("Enter your LangSmith API Key: ")

# Option 2: Direct assignment (replace with your actual keys)
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPEN_AI_KEY_NEW')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "default"

# Verify packages loaded correctly
import langchain
import openai
print(f"✅ LangChain version: {langchain.__version__}")
print(f"✅ OpenAI version: {openai.__version__}")
print("✅ API keys configured!")

✅ LangChain version: 0.3.7
✅ OpenAI version: 1.54.0
✅ API keys configured!


## Step 3: Load Data

In [ ]:
import pandas as pd

# Download the medical chatbot dataset
!gdown 1FpC7_DaxWQJf4JoVQLlbSYcSTZSz86C6 -q

df = pd.read_csv("ai-medical-chatbot.csv")
print(f"✅ Loaded {len(df)} records")
df.head()

✅ Loaded 256916 records


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [ ]:
# Combine the fields into one document for embedding
df['document'] = (
    df['Description'].fillna('') + '\n\n' +
    'Patient: ' + df['Patient'].fillna('') + '\n\n' +
    'Doctor: ' + df['Doctor'].fillna('')
)

print("Sample document:")
print("-" * 50)
print(df['document'][0])

Sample document:
--------------------------------------------------
Q. What does abutment of the nerve root mean?

Patient: Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bulging and tear?

Doctor: Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->


## Step 4: Create Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.docstore.document import Document
import os
import shutil

# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Set up the vector store
PERSIST_DIR = "chroma_db"

# Clean up existing DB to ensure fresh start
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

# Create documents from the dataframe (using first 75 records)
documents = [Document(page_content=row['document']) for _, row in df.head(75).iterrows()]

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)

# Set up retriever
retriever = vectorstore.as_retriever()

print(f"✅ Vector store created with {vectorstore._collection.count()} documents")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store created with 75 documents


## Step 5: Create RAG Chain

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_openai import ChatOpenAI
from langchain.tools import Tool

# Define the system prompt
system_prompt = (
    "You are a helpful medical assistant. Answer the user's question based on the context provided below. "
    "If the context contains relevant information, use it to provide a clear answer. "
    "If you cannot find the answer in the context, simply say you don't have that information. "
    "Be concise and helpful."
    "\n\n"
    "{context}"
)

# Create a chat prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# Initialize the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2
)

# Create the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, rag_prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Create a tool function for the agent
def rag_context_retriever(user_input: str) -> str:
    """Retrieves relevant RAG context given user input."""
    try:
        rag_response = rag_chain.invoke({"input": user_input})
        return rag_response.get("answer", "Sorry, no RAG answer found.")
    except Exception as e:
        return f"Error retrieving RAG context: {e}"

# Create the tool
rag_context_tool = Tool(
    name="RAG_Context_Retriever",
    func=rag_context_retriever,
    description="Retrieves relevant RAG context given user input and chat history."
)

print("✅ RAG chain and tool created!")

✅ RAG chain and tool created!


In [ ]:
rag_context_tool.invoke('hi')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'Hello! How can I assist you today?'

## Step 6: Create Agent

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Set up memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Your tools
tools = [rag_context_tool]

# Agent prompt template
template = """You are a helpful medical assistant. Get an answer based only on the internal (RAG).
Do not answer it on your own.

If the question is non-medical, respond:
Final Answer: I'm not allowed to discuss it.

Tools:
{tools}

Use the following format:

Question: {input}

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

prompt = PromptTemplate(
    input_variables=["input", "tools", "tool_names", "agent_scratchpad"],
    template=template
)

# Create the agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# Create the agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

print("✅ Agent created!")

✅ Agent created!


/tmp/ipython-input-1906134052.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


## Step 7: Test the Agent

In [ ]:
# Test with a simple greeting
response = agent_executor.invoke({"input": "hi"})
print("\nResponse:", response['output'])



> Entering new AgentExecutor chain...
Action: RAG_Context_Retriever  
Action Input: "hi"  Hello! How can I assist you today?Final Answer: I'm not allowed to discuss it.

> Finished chain.

Response: I'm not allowed to discuss it.


In [ ]:
# Test with a medical question
response = agent_executor.invoke({"input": "what are the causes of chest pain"})
print("\nResponse:", response['output'])



> Entering new AgentExecutor chain...
Action: RAG_Context_Retriever  
Action Input: "what are the causes of chest pain"  Chest pain can have various causes, including:

1. **Cardiac Issues**: Conditions such as angina, heart attack, or pericarditis can cause chest pain. Cardiac pain is often severe and may radiate to the left arm.

2. **Gastrointestinal Problems**: Acid reflux, esophagitis, or gallbladder issues can lead to chest discomfort.

3. **Musculoskeletal Causes**: Muscle strain, rib injuries, or conditions like costochondritis can result in localized chest pain.

4. **Pulmonary Conditions**: Issues such as pneumonia, pleuritis, or pulmonary embolism can cause chest pain, often accompanied by breathing difficulties.

5. **Anxiety and Stress**: Emotional distress can lead to chest pain that is often described as tightness or pressure.

6. **Other Causes**: Conditions like shingles or aortic dissection can also present with chest pain.

It's important to differentiate between t

## Step 8: Set Up LangSmith Evaluation

In [ ]:
from langsmith import Client
from langchain.evaluation import load_evaluator

# Initialize LangSmith client
client = Client()

print("✅ LangSmith client initialized!")

✅ LangSmith client initialized!


In [ ]:
# Create prediction wrapper function
def predict(inputs: dict) -> dict:
    """Wrapper function that takes inputs and returns prediction"""
    response = agent_executor.invoke({"input": inputs["question"]})
    return {"answer": response["output"]}

## Step 9: Define Evaluation Metrics

In [ ]:
# 1. Answer Correctness Evaluator
correctness_evaluator = load_evaluator(
    "labeled_criteria",
    criteria={
        "correctness": """
        Evaluate if the submission is correct, accurate, and factual compared to the reference answer.

        Consider the submission CORRECT (Y) if:
          Main facts align with the reference answer
          Core meaning is the same (even if worded differently)
          No significant factual errors
          Both refuse to answer (different refusal wording is OK)
          Minor paraphrasing or reasonable additions are acceptable

        Consider the submission INCORRECT (N) if:
          Contains factual errors or contradicts reference
          Misses key information from reference
          Provides answer when should refuse (or vice versa)
          Significantly different meaning from reference

        Special cases for medical chatbot:
        - "I don't have information" ≈ "I'm not allowed to discuss it" → CORRECT
        - Both refuse to answer = CORRECT (semantically equivalent)
        - One provides info, one refuses = INCORRECT (mismatch)

        Focus on semantic equivalence, not exact word matching.
        """
    },
    llm=llm
)

print("✅ Correctness evaluator created!")

✅ Correctness evaluator created!


In [ ]:
# 2. Answer Hallucination Evaluator
hallucination_evaluator = load_evaluator(
    "criteria",
    criteria={
        "hallucination": """
        Evaluate the degree of hallucination in the answer.

        Rate on a scale where higher is BETTER:

        CRITICAL: Return a score between 0.0 and 1.0

        0.0-0.3 (Severe): Most information is fabricated or unsupported
        0.4-0.6 (Moderate): Mix of factual and fabricated content
        0.7-0.8 (Minor): Mostly accurate with small embellishments
        0.9-1.0 (None): Completely grounded, no fabrication

        In your reasoning, EXPLICITLY state the score as a decimal between 0.0 and 1.0
        """
    },
    llm=llm
)

print("✅ Hallucination evaluator created!")

✅ Hallucination evaluator created!


In [ ]:
# 3. Document Relevance Evaluator
relevance_evaluator = load_evaluator(
    "criteria",
    criteria={
        "relevance": """
        Evaluate if the answer is RELEVANT to addressing the user's question.

        The answer should:
        - Directly address what the user asked
        - Provide on-topic information
        - Be helpful and actionable for the question

        Consider RELEVANT (Y) if:
        - Answer directly addresses the question
        - Information is on-topic and useful
        - May have minor gaps but core question is answered

        Consider IRRELEVANT (N) if:
        - Answer is off-topic or unrelated
        - Misses the main point of the question
        - Provides generic info not applicable to the question

        Answer Y if: Answer is relevant to the question
        Answer N if: Answer is not relevant to the question
        """
    },
    llm=llm
)

print("✅ Relevance evaluator created!")

✅ Relevance evaluator created!


## Step 10: Create Evaluation Dataset

In [ ]:
# Sample test cases with ground truth answers
evaluation_dataset = [
    {
        "question": "What are the causes of chest pain?",
        "ground_truth": "Chest pain can be caused by cardiac issues, gastrointestinal problems, musculoskeletal issues, or pulmonary conditions."
    },
    {
        "question": "What does abutment of nerve root mean?",
        "ground_truth": "Abutment of nerve root refers to compression or contact with the nerve root in spinal conditions."
    },
    {
        "question": "How to treat acne?",
        "ground_truth": "Acne treatment includes topical medications, oral medications, lifestyle changes, and proper skincare."
    },
    {
        "question": "What is Diabtese?",
        "ground_truth": "I don't have context about this."
    }
]

print(f"✅ Created evaluation dataset with {len(evaluation_dataset)} examples")

✅ Created evaluation dataset with 4 examples


In [ ]:
# Create dataset in LangSmith
import uuid

# Use a unique dataset name to avoid conflicts
dataset_name = f"medical-chatbot-eval-{uuid.uuid4().hex[:8]}"

try:
    # Create new dataset
    dataset = client.create_dataset(dataset_name)

    # Add examples to dataset
    for example in evaluation_dataset:
        client.create_example(
            inputs={"question": example["question"]},
            outputs={"answer": example["ground_truth"]},
            dataset_id=dataset.id
        )
    print(f"✅ Created dataset: {dataset_name}")
except Exception as e:
    print(f"⚠️ Error creating dataset: {e}")
    # Try to use existing dataset
    try:
        dataset = client.read_dataset(dataset_name=dataset_name)
        print(f"✅ Using existing dataset: {dataset_name}")
    except:
        print("❌ Could not create or find dataset. Check your LangSmith API key.")

✅ Created dataset: medical-chatbot-eval-9c9f771d


## Step 11: Define Custom Evaluator Functions

In [ ]:
def evaluate_correctness(run, example):
    """Evaluate answer correctness against ground truth"""
    result = correctness_evaluator.evaluate_strings(
        prediction=run.outputs["answer"],
        reference=example.outputs["answer"],
        input=example.inputs["question"]
    )
    return {
        "score": result["score"],
        "reasoning": result.get("reasoning", ""),
        "value": result.get("value", "")
    }


def evaluate_hallucination(run, example):
    """Evaluate if answer contains hallucinations"""
    result = hallucination_evaluator.evaluate_strings(
        prediction=run.outputs["answer"],
        input=example.inputs["question"]
    )
    return {
        "score": result["score"],
        "reasoning": result.get("reasoning", "")
    }


def evaluate_relevance(run, example):
    """Evaluate relevance of answer to question"""
    result = relevance_evaluator.evaluate_strings(
        prediction=run.outputs["answer"],
        input=example.inputs["question"]
    )
    return {
        "score": result["score"],
        "reasoning": result.get("reasoning", ""),
        "value": result.get("value", "")
    }


print("✅ Custom evaluator functions defined!")

✅ Custom evaluator functions defined!


## Step 12: Run Full Evaluation

In [ ]:
from langsmith import evaluate

print("\n" + "=" * 50)
print("RUNNING RAG EVALUATION")
print("=" * 50 + "\n")

# Run evaluation
results = evaluate(
    predict,
    data=dataset_name,
    evaluators=[
        evaluate_correctness,
        evaluate_hallucination,
        evaluate_relevance
    ],
    experiment_prefix="medical-chatbot-eval",
    metadata={
        "model": "gpt-4o-mini",
        "version": "1.0"
    }
)

print("\n✅ Evaluation complete! Check LangSmith for detailed results.")


RUNNING RAG EVALUATION

View the evaluation results for experiment: 'medical-chatbot-eval-9eef7867' at:
https://smith.langchain.com/o/857ac937-0557-4eac-b464-d8a52184783e/datasets/6ec1f9df-50cd-4185-ac60-bf960e7eea0d/compare?selectedSessions=fd64afc0-b527-4a03-9930-644b0bd5dd5b




0it [00:00, ?it/s]



> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...
Action: RAG_Context_Retriever  
Action Input: "What does abutment of nerve root mean?"  Action: RAG_Context_Retriever  
Action Input: "What are the causes of chest pain?"  Action: RAG_Context_Retriever  
Action Input: "How to treat acne?"  Action: RAG_Context_Retriever  
Action Input: "What is Diabtese?"  I don't have that information.Final Answer: I'm not allowed to discuss it.

> Finished chain.
Abutment of the nerve root refers to a situation where a structure, such as a herniated disc or bone spur, is in close proximity to or pressing against a nerve root in the spine. This can potentially lead to nerve irritation or compression, resulting in pain, numbness, or weakness in the areas served by that nerve. For specific treatment options related to your condition, it's best to consult a healthcare professional.To treat acne, it

## Step 13: Test Individual Evaluation (Optional)

In [ ]:
print("\n" + "=" * 50)
print("TESTING INDIVIDUAL EVALUATION")
print("=" * 50 + "\n")

# Test with a single example
test_question = "What are the causes of chest pain?"
test_response = agent_executor.invoke({"input": test_question})

print(f"Question: {test_question}")
print(f"\nAgent Response: {test_response['output']}")

# Evaluate this response
print("\n--- Correctness Evaluation ---")
correctness_result = correctness_evaluator.evaluate_strings(
    prediction=test_response['output'],
    reference="Chest pain can be caused by cardiac issues, gastrointestinal problems, musculoskeletal issues, or pulmonary conditions.",
    input=test_question
)
print(f"Score: {correctness_result['score']}")
print(f"Reasoning: {correctness_result.get('reasoning', 'N/A')}")

print("\n--- Hallucination Check ---")
hallucination_result = hallucination_evaluator.evaluate_strings(
    prediction=test_response['output'],
    input=test_question
)
print(f"Score: {hallucination_result['score']}")
print(f"Reasoning: {hallucination_result.get('reasoning', 'N/A')}")

print("\n--- Relevance Check ---")
relevance_result = relevance_evaluator.evaluate_strings(
    prediction=test_response['output'],
    input=test_question
)
print(f"Score: {relevance_result['score']}")
print(f"Reasoning: {relevance_result.get('reasoning', 'N/A')}")


TESTING INDIVIDUAL EVALUATION



> Entering new AgentExecutor chain...
Action: RAG_Context_Retriever  
Action Input: "causes of chest pain"  Chest pain can have various causes, including:

1. **Cardiac Issues**: Conditions like angina, heart attack, or pericarditis can cause chest pain. Cardiac pain is often severe and may radiate to the left arm.

2. **Gastrointestinal Problems**: Acid reflux, esophageal spasms, or gastritis can lead to chest discomfort.

3. **Musculoskeletal Issues**: Strain or injury to the chest muscles or ribs can cause pain, often worsened by movement or certain positions.

4. **Pulmonary Conditions**: Issues like pneumonia, pleuritis, or pulmonary embolism can result in chest pain, often accompanied by breathing difficulties.

5. **Anxiety and Stress**: Psychological factors can lead to chest pain that mimics cardiac issues.

If you are experiencing chest pain, especially if it is severe or accompanied by other symptoms, it is important to seek medical attentio

## Summary

This notebook demonstrates:
1. Building a RAG pipeline with LangChain
2. Creating a medical chatbot agent
3. Evaluating the chatbot using LangSmith with custom metrics:
   - **Correctness**: Does the answer match the ground truth?
   - **Hallucination**: Is the answer grounded in facts?
   - **Relevance**: Does the answer address the question?

Check your [LangSmith dashboard](https://smith.langchain.com/) to see detailed evaluation results!